# Function 5: 4D Unimodal Process Optimisation
**Programme:** Imperial College London – Professional Certificate in Machine Learning & AI  
**Domain Context:** Vaccine Bioreactor Yield Optimisation (4 Continuous Control Parameters)  
**Author:** Dr. Joseph Neary  

---

## 1. Problem Framing & Objective
Function 5 accepts a 4D feature vector $\mathbf{x} = (x_1, x_2, x_3, x_4)$ bounded within $[0, 1]^4$. The objective is to **maximise** the scalar output $y$, which represents chemical/bioprocess yield. 

According to problem design, this surface is **unimodal** (possessing a single dominant global maximum). We deploy a **Gaussian Process (GP) Surrogate Model** using a smooth Radial Basis Function (RBF) kernel combined with target scaling (`normalize_y=True`) and noise regularization ($\alpha = 10^{-3}$). We evaluate candidate utility across a $25 \times 25 \times 25 \times 25$ evaluation hypercube ($390,625$ points) using the **Upper Confidence Bound (UCB)** acquisition function.

## 2. Ingestion & Baseline Data Analysis
We ingest the $N = 10$ seed observations provided in `Data/Raw/function_5/` and display them sorted by descending target output $y$.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

# ==========================================
# 1. LOAD DATA FROM YOUR RAW DATA DIRECTORY
# ==========================================
base_dir = r"C:\Projects\ML\capstone\Imperial-ML-Capstone-Project"

input_path = os.path.join(base_dir, 'Data', 'Raw', 'function_5', 'initial_inputs.npy')
output_path = os.path.join(base_dir, 'Data', 'Raw', 'function_5', 'initial_outputs.npy')

X_init = np.load(input_path)   # Shape: (10, 4) - 4D input coordinates
y_init = np.load(output_path)  # Shape: (10,)   - Target score

# Display initial data table sorted by current peak score
df_f5 = pd.DataFrame(X_init, columns=['x1', 'x2', 'x3', 'x4'])
df_f5['y'] = y_init

print("=== Function 5: Initial Observations (Sorted High to Low) ===")
print(df_f5.sort_values(by='y', ascending=False).reset_index(drop=True))
print("\n" + "="*60 + "\n")

=== Function 5: Initial Observations (Sorted High to Low) ===
          x1        x2        x3        x4            y
0   0.224189  0.846480  0.879484  0.878516  1088.859618
1   0.119879  0.862540  0.643331  0.849804   431.612757
2   0.438933  0.774092  0.378167  0.933696   355.806818
3   0.836478  0.193610  0.663893  0.785649   258.370525
4   0.463442  0.630025  0.107906  0.957644   233.223610
5   0.352356  0.322242  0.116979  0.473113   109.571876
6   0.511142  0.817957  0.728710  0.112354    79.729130
7   0.683432  0.118663  0.829046  0.567577    78.434389
8   0.191447  0.038193  0.607418  0.414584    64.443440
9   0.583973  0.147243  0.348097  0.428615    64.420147
10  0.306889  0.316878  0.622634  0.095399    63.476716
11  0.553621  0.667350  0.323806  0.814870    57.571537
12  0.355482  0.639619  0.417618  0.122604    45.181570
13  0.725262  0.479870  0.088947  0.759760    28.866752
14  0.677491  0.358510  0.479592  0.072880    24.423088
15  0.758653  0.536518  0.656000  0.360342

## 3. GP Surrogate Fitting & 4D Grid Evaluation
We fit a non-parametric GP regressor with an RBF kernel:

$$k(\mathbf{x}, \mathbf{x}') = C \cdot \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

We evaluate the **Upper Confidence Bound (UCB)** acquisition utility across the 4D search lattice:

$$\text{UCB}(\mathbf{x}) = \mu(\mathbf{x}) + \kappa \cdot \sigma(\mathbf{x})$$

Where $\kappa = 2.0$ balances expected yield exploitation ($\mu$) with spatial exploration ($\sigma$).

In [2]:
# ==========================================
# 2. FIT GAUSSIAN PROCESS (GP) SURROGATE MODEL
# ==========================================
kernel = C(1.0, (1e-5, 1e5)) * RBF(length_scale=0.5, length_scale_bounds=(1e-3, 1e2))

gp = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=10, 
    alpha=1e-3,          # Regularization for noise tolerance
    normalize_y=True,    # Standardises target y values
    random_state=42
)

# Fit GP model to 4D inputs
gp.fit(X_init, y_init)

print("=== Fitted Kernel Hyperparameters ===")
print(gp.kernel_)
print("="*60 + "\n")

# ==========================================
# 3. 4D GRID SEARCH FOR UCB MAXIMISATION
# ==========================================
grid_res = 25  # 25^4 = 390,625 evaluation points
x1_dom = np.linspace(0, 1, grid_res)
x2_dom = np.linspace(0, 1, grid_res)
x3_dom = np.linspace(0, 1, grid_res)
x4_dom = np.linspace(0, 1, grid_res)

X1_m, X2_m, X3_m, X4_m = np.meshgrid(x1_dom, x2_dom, x3_dom, x4_dom)
X_test = np.column_stack([X1_m.ravel(), X2_m.ravel(), X3_m.ravel(), X4_m.ravel()])

# Predict posterior mean and standard deviation across 4D space
mu, sigma = gp.predict(X_test, return_std=True)

# Upper Confidence Bound (UCB)
kappa = 2.0
ucb_values = mu + kappa * sigma

best_idx = np.argmax(ucb_values)
next_query = X_test[best_idx]

# ==========================================
# 4. PORTAL SUBMISSION FORMATTING
# ==========================================
x1_q = round(next_query[0], 6)
x2_q = round(next_query[1], 6)
x3_q = round(next_query[2], 6)
x4_q = round(next_query[3], 6)

print("=== RECOMMENDED SUBMISSION FOR FUNCTION 5 (WEEK 1) ===")
print(f"Coordinates (x1, x2, x3, x4): [{x1_q}, {x2_q}, {x3_q}, {x4_q}]")
print(f"Portal Submission Format:  {x1_q:.6f} - {x2_q:.6f} - {x3_q:.6f} - {x4_q:.6f}")
print("======================================================")

=== Fitted Kernel Hyperparameters ===
1.15**2 * RBF(length_scale=0.318)

=== RECOMMENDED SUBMISSION FOR FUNCTION 5 (WEEK 1) ===
Coordinates (x1, x2, x3, x4): [0.375, 0.833333, 1.0, 1.0]
Portal Submission Format:  0.375000 - 0.833333 - 1.000000 - 1.000000


## Imperial Capstone Reflection: Function 5 (Week 1)

**Query Input Coordinates:** `0.375000 - 0.833333 - 1.000000 - 1.000000`

---

### 1. What method was used and why?
I implemented a **Gaussian Process (GP) Regression** surrogate model paired with an **Upper Confidence Bound (UCB)** acquisition function evaluated over a $25 \times 25 \times 25 \times 25$ grid ($390,625$ candidate points) across the 4D parameter hypercube $[0, 1]^4$.

Because Function 5 is characterized by a **unimodal** response surface (featuring a single dominant global peak representing optimal bioprocess yield), a non-parametric GP surrogate model with a smooth Radial Basis Function (RBF) kernel was selected:

$$k(\mathbf{x}, \mathbf{x}') = C \cdot \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

Target normalization (`normalize_y=True`) and noise regularization ($\alpha = 10^{-3}$) were applied to stabilize the Maximum Marginal Likelihood (MML) hyperparameter tuning. The fitted kernel converged to a variance scale of $C = 1.15^2$ and a length scale of $\ell = 0.318$.

---

### 2. Was the query focused on exploration or exploitation?
The query represents a **balanced exploration-exploitation strategy** using $\kappa = 2.0$ in the UCB utility formulation:

$$\text{UCB}(\mathbf{x}) = \mu(\mathbf{x}) + \kappa \cdot \sigma(\mathbf{x})$$

Although Function 5 is unimodal, the initial baseline data consists of only $N = 10$ scattered points, leaving large expanses of the 4D space unmapped. Setting $\kappa = 2.0$ allows the optimizer to balance exploiting the highest current mean predictions ($\mu$) with systematically exploring high-uncertainty boundaries ($\sigma$) to locate the dominant peak without premature convergence.

---

### 3. What did the baseline dataset teach us about the underlying function?
The fitted length scale ($\ell = 0.318$) indicates that the response surface transitions smoothly over moderate input distances. Because the space is unimodal, the Gaussian Process successfully smooths out minor localized noise and constructs a coherent predictive map toward the upper bounds of features $x_3$ and $x_4$.

---

### 4. What is the strategy for the next round?
Upon receiving the true experimental yield score $y_{\text{new}}$ for `0.375000 - 0.833333 - 1.000000 - 1.000000`, I will append this 11th point to the dataset and re-fit the GP model:
* **If $y_{\text{new}}$ yields a strong positive improvement:** Decrease $\kappa \to 1.0$ to aggressively exploit the immediate neighborhood of these coordinates.
* **If $y_{\text{new}}$ is lower than anticipated:** Retain $\kappa = 2.0$ to continue mapping the broader 4D hypercube until the true global maximum is decisively isolated.